In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from collections import Counter

# Charger les données
df = pd.read_csv('../data/raw/alzheimers_disease_data.csv')

# Supprimer les colonnes inutiles
df = df.drop(columns=['PatientID', 'DoctorInCharge'])

print(f'Shape : {df.shape}')
print(f'Colonnes : {df.columns.tolist()}')

Shape : (2149, 33)
Colonnes : ['Age', 'Gender', 'Ethnicity', 'EducationLevel', 'BMI', 'Smoking', 'AlcoholConsumption', 'PhysicalActivity', 'DietQuality', 'SleepQuality', 'FamilyHistoryAlzheimers', 'CardiovascularDisease', 'Diabetes', 'Depression', 'HeadInjury', 'Hypertension', 'SystolicBP', 'DiastolicBP', 'CholesterolTotal', 'CholesterolLDL', 'CholesterolHDL', 'CholesterolTriglycerides', 'MMSE', 'FunctionalAssessment', 'MemoryComplaints', 'BehavioralProblems', 'ADL', 'Confusion', 'Disorientation', 'PersonalityChanges', 'DifficultyCompletingTasks', 'Forgetfulness', 'Diagnosis']


In [2]:
# Séparer les features et la cible
X = df.drop(columns=['Diagnosis'])
y = df['Diagnosis']

print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')
print(f'Distribution cible : {Counter(y)}')

X shape : (2149, 32)
y shape : (2149,)
Distribution cible : Counter({0: 1389, 1: 760})


In [ ]:
# Split 70% train / 15% validation / 15% test

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp
)

print(f'Train : {X_train.shape} — {Counter(y_train)}')
print(f'Val   : {X_val.shape} — {Counter(y_val)}')
print(f'Test  : {X_test.shape} — {Counter(y_test)}')

Train : (1503, 32) — Counter({0: 971, 1: 532})
Val   : (323, 32) — Counter({0: 209, 1: 114})
Test  : (323, 32) — Counter({0: 209, 1: 114})


In [4]:
# Normalisation des colonnes continues
continuous_cols = ['Age', 'BMI', 'AlcoholConsumption', 'PhysicalActivity',
                   'DietQuality', 'SleepQuality', 'SystolicBP', 'DiastolicBP',
                   'CholesterolTotal', 'CholesterolLDL', 'CholesterolHDL',
                   'CholesterolTriglycerides', 'MMSE', 'FunctionalAssessment', 'ADL']

scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_val[continuous_cols]   = scaler.transform(X_val[continuous_cols])
X_test[continuous_cols]  = scaler.transform(X_test[continuous_cols])

print('Normalisation OK')
print(f'Moyenne Age train (doit être ~0) : {X_train["Age"].mean():.4f}')
print(f'Std Age train (doit être ~1) : {X_train["Age"].std():.4f}')

Normalisation OK
Moyenne Age train (doit être ~0) : -0.0000
Std Age train (doit être ~1) : 1.0003


In [5]:
# SMOTE sur le train set uniquement
print(f'Avant SMOTE : {Counter(y_train)}')

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f'Après SMOTE : {Counter(y_train_bal)}')
print(f'Shape train final : {X_train_bal.shape}')

Avant SMOTE : Counter({0: 971, 1: 532})
Après SMOTE : Counter({0: 971, 1: 971})
Shape train final : (1942, 32)


In [6]:
# Sauvegarder les splits et le scaler
joblib.dump(
    (X_train_bal, X_val, X_test,
     y_train_bal, y_val, y_test,
     scaler, list(X_train.columns)),
    '../data/processed/splits.pkl'
)
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(list(X_train.columns), '../models/feature_names.pkl')

print('Tout sauvegardé :')
print('  - data/processed/splits.pkl')
print('  - models/scaler.pkl')
print('  - models/feature_names.pkl')

Tout sauvegardé :
  - data/processed/splits.pkl
  - models/scaler.pkl
  - models/feature_names.pkl
